In [1]:
import pandas as pd

In [12]:
cluster_id_data = pd.read_excel(r"ID_Group_result.xlsx")
inventory_data = pd.read_excel(r"W33836_business_analytics_detailed_purchase_history.xlsx", usecols=["Purchase_Date", "ID", "Product", "Amount"])

In [13]:
inventory_data

,Purchase_Date,ID,Product,Amount
0,2012-01-01,375,Fish,29
1,2012-01-01,387,Meat,6
2,2012-01-01,679,Fruits,2
3,2012-01-01,819,Meat,83
4,2012-01-01,1160,Meat,1
...,...,...,...,...
28832,2014-12-31,9938,Fruits,7
28833,2014-12-31,10095,Fruits,3
28834,2014-12-31,10270,Wines,5
28835,2014-12-31,10339,Gold,6


In [14]:
inventory_data.duplicated().sum()

np.int64(0)

In [16]:
merge_data = pd.merge(inventory_data, cluster_id_data, on="ID", how="left")
merge_data.isna().sum()

Purchase_Date      0
ID                 0
Product            0
Amount             0
Cluster_group    108
dtype: int64

In [26]:
no_cluster_data = merge_data.query("Cluster_group.isnull()")
no_cluster_data

,Purchase_Date,ID,Product,Amount,Cluster_group
40,2012-01-02,492,Gold,22,NaN
373,2012-01-14,9432,Fish,4,NaN
510,2012-01-19,7829,Wines,5,NaN
769,2012-01-28,11004,Gold,2,NaN
1189,2012-02-13,492,Meat,9,NaN
...,...,...,...,...,...
27540,2014-11-07,11133,Gold,22,NaN
27543,2014-11-08,492,Wines,103,NaN
28151,2014-12-03,7829,Wines,6,NaN
28164,2014-12-04,1150,Fish,33,NaN


In [28]:
# The deleted data 
no_cluster_data["ID"].unique()

array([  492,  9432,  7829, 11004, 11133,  1150,  7734,  4369])

In [29]:
cleaned_merge_data = merge_data.query("not Cluster_group.isnull()")
cleaned_merge_data

,Purchase_Date,ID,Product,Amount,Cluster_group
0,2012-01-01,375,Fish,29,2.0
1,2012-01-01,387,Meat,6,1.0
2,2012-01-01,679,Fruits,2,1.0
3,2012-01-01,819,Meat,83,0.0
4,2012-01-01,1160,Meat,1,1.0
...,...,...,...,...,...
28832,2014-12-31,9938,Fruits,7,2.0
28833,2014-12-31,10095,Fruits,3,2.0
28834,2014-12-31,10270,Wines,5,1.0
28835,2014-12-31,10339,Gold,6,1.0


In [30]:
cleaned_merge_data.isna().sum()

Purchase_Date    0
ID               0
Product          0
Amount           0
Cluster_group    0
dtype: int64

# Change Purchase_Date's granularity

In [35]:
cleaned_merge_data["Purchase_Date"] = pd.to_datetime(cleaned_merge_data["Purchase_Date"])
cleaned_merge_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28729 entries, 0 to 28836
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Purchase_Date  28729 non-null  datetime64[ns]
 1   ID             28729 non-null  int64         
 2   Product        28729 non-null  object        
 3   Amount         28729 non-null  int64         
 4   Cluster_group  28729 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(2), object(1)
memory usage: 1.3+ MB


C:\Users\user\AppData\Local\Temp\ipykernel_24940\869130475.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_merge_data["Purchase_Date"] = pd.to_datetime(cleaned_merge_data["Purchase_Date"])


In [37]:
cleaned_merge_data["Purchase_Date_Month"] = cleaned_merge_data["Purchase_Date"].dt.to_period('M')
cleaned_merge_data

C:\Users\user\AppData\Local\Temp\ipykernel_24940\3717823417.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_merge_data["Purchase_Date_Month"] = cleaned_merge_data["Purchase_Date"].dt.to_period('M')


,Purchase_Date,ID,Product,Amount,Cluster_group,Purochase_Date_Month,Purchase_Date_Month
0,2012-01-01,375,Fish,29,2.0,2012-01,2012-01
1,2012-01-01,387,Meat,6,1.0,2012-01,2012-01
2,2012-01-01,679,Fruits,2,1.0,2012-01,2012-01
3,2012-01-01,819,Meat,83,0.0,2012-01,2012-01
4,2012-01-01,1160,Meat,1,1.0,2012-01,2012-01
...,...,...,...,...,...,...,...
28832,2014-12-31,9938,Fruits,7,2.0,2014-12,2014-12
28833,2014-12-31,10095,Fruits,3,2.0,2014-12,2014-12
28834,2014-12-31,10270,Wines,5,1.0,2014-12,2014-12
28835,2014-12-31,10339,Gold,6,1.0,2014-12,2014-12


In [39]:
final_cleaned_data = cleaned_merge_data[['Purchase_Date_Month', "Product", "Cluster_group", 'Amount']]
final_cleaned_data

,Purchase_Date_Month,Product,Cluster_group,Amount
0,2012-01,Fish,2.0,29
1,2012-01,Meat,1.0,6
2,2012-01,Fruits,1.0,2
3,2012-01,Meat,0.0,83
4,2012-01,Meat,1.0,1
...,...,...,...,...
28832,2014-12,Fruits,2.0,7
28833,2014-12,Fruits,2.0,3
28834,2014-12,Wines,1.0,5
28835,2014-12,Gold,1.0,6


# Groupby [Month, Product, Cluster]

In [44]:
group_by_data = final_cleaned_data.groupby(
    ["Cluster_group", "Product", "Purchase_Date_Month"]).agg({"Amount":"sum"}
)

group_by_data

Amount
Cluster_group Product Purchase_Date_Month        
0.0           Fish    2012-01                 501
                      2012-02                 761
                      2012-03                 424
                      2012-04                 641
                      2012-05                 385
...                                           ...
2.0           Wines   2014-08               13284
                      2014-09               13438
                      2014-10               10594
                      2014-11               14707
                      2014-12               12227

[540 rows x 1 columns]

In [47]:
group_by_data.to_excel("Cleaned_Inventory_use_data.xlsx", index=True)